# 🐍 Clase 5 · OOP II — OrderBook y PositionTracker

> Construir el libro como objeto que contiene niveles, con métricas como métodos. Y un PositionTracker que consume objetos Fill. Aquí ves cómo los objetos se entrelazan.

**Hoy construyes:** clases OrderBook y PositionTracker (composición).

⏱️ 🟢 núcleo ~15 min · 🔵 si vamos bien +18 min.

### Cómo funciona este cuaderno

1. Escribe tu respuesta en la celda de código.
2. Debajo hay una **✅ comprobación plegada**: ejecútala con `Shift+Enter` para validarte (despliégala si quieres ver el `assert`).
3. ¿Atascado? Algunos ejercicios traen una **💭 Pista** intermedia; si no basta, abre **💡 Ver solución**.

Cada ejercicio lleva su etiqueta: **🟢 núcleo** (en clase) · **🔵 si vamos bien** · **🟣 bonus** (el cuaderno de auxiliares profundiza más).

### 1. OrderBook: un objeto que contiene niveles

<sub>🟢 núcleo · ~5 min</sub>

Define `OrderBook(bids, asks)` donde cada lado es una lista de tuplas `(price, size)`. El libro **contiene** sus niveles como atributos.

<sub>practicas: composición</sub>

In [ ]:
class OrderBook:
    def __init__(self, bids, asks):
        pass

In [ ]:
# ✅ Comprobación — ejecútala (Shift+Enter). Está plegada a propósito.
assert OrderBook.__init__.__code__.co_consts != (None,), '⏸ implementa OrderBook.__init__: su cuerpo sigue siendo pass'
b = OrderBook([(100,1)], [(101,2)])
assert b.bids == [(100,1)] and b.asks == [(101,2)]
print('ok')

<details>
<summary>💡 Ver solución</summary>

```python
class OrderBook:
    def __init__(self, bids, asks):
        self.bids = bids
        self.asks = asks
```

</details>

### 2. best_bid / best_ask / spread / mid

<sub>🟢 núcleo · ~5 min</sub>

Reescribe `OrderBook` con métodos `best_bid()`, `best_ask()`, `spread()` y `mid()`. (Ordena bids desc y asks asc en `__init__`; el mejor es el primero.) Las funciones de la clase 2 ahora son **métodos**.

> 💡 `self.bids = sorted(bids, key=lambda x: -x[0])`.

<sub>practicas: métodos sobre el estado</sub>

In [ ]:
class OrderBook:
    def __init__(self, bids, asks):
        self.bids = sorted(bids, key=lambda x: -x[0])
        self.asks = sorted(asks, key=lambda x: x[0])
    def best_bid(self):
        pass
    def best_ask(self):
        pass
    def spread(self):
        pass
    def mid(self):
        pass

In [ ]:
# ✅ Comprobación — ejecútala (Shift+Enter). Está plegada a propósito.
assert OrderBook.best_bid.__code__.co_consts != (None,), '⏸ implementa OrderBook.best_bid: su cuerpo sigue siendo pass'
assert OrderBook.best_ask.__code__.co_consts != (None,), '⏸ implementa OrderBook.best_ask: su cuerpo sigue siendo pass'
assert OrderBook.spread.__code__.co_consts != (None,), '⏸ implementa OrderBook.spread: su cuerpo sigue siendo pass'
assert OrderBook.mid.__code__.co_consts != (None,), '⏸ implementa OrderBook.mid: su cuerpo sigue siendo pass'
b = OrderBook([(100,1),(99,1)], [(101,1),(102,1)])
assert b.best_bid()==100 and b.best_ask()==101
assert b.spread()==1 and b.mid()==100.5
print('ok')

<details>
<summary>💡 Ver solución</summary>

```python
class OrderBook:
    def __init__(self, bids, asks):
        self.bids = sorted(bids, key=lambda x: -x[0])
        self.asks = sorted(asks, key=lambda x: x[0])
    def best_bid(self):
        return self.bids[0][0]
    def best_ask(self):
        return self.asks[0][0]
    def spread(self):
        return self.best_ask() - self.best_bid()
    def mid(self):
        return (self.best_bid() + self.best_ask()) / 2
```

</details>

### 3. imbalance() del nivel 1

<sub>🟢 núcleo · ~5 min</sub>

Añade a `OrderBook` el método `imbalance()` = (bid_size − ask_size)/(bid_size + ask_size) en el mejor nivel.

<sub>practicas: otro método</sub>

In [ ]:
class OrderBook:
    def __init__(self, bids, asks):
        self.bids = sorted(bids, key=lambda x: -x[0])
        self.asks = sorted(asks, key=lambda x: x[0])
    def imbalance(self):
        pass

In [ ]:
# ✅ Comprobación — ejecútala (Shift+Enter). Está plegada a propósito.
assert OrderBook.imbalance.__code__.co_consts != (None,), '⏸ implementa OrderBook.imbalance: su cuerpo sigue siendo pass'
b = OrderBook([(100,3)], [(101,1)])
assert abs(b.imbalance() - 0.5) < 1e-9
print('ok')

<details>
<summary>💭 Pista (antes de mirar la solución)</summary>

El mejor nivel es el primero tras ordenar: `self.bids[0]` y `self.asks[0]` son tuplas `(precio, tamaño)`, así que el tamaño es el índice `[1]`.

</details>

<details>
<summary>💡 Ver solución</summary>

```python
class OrderBook:
    def __init__(self, bids, asks):
        self.bids = sorted(bids, key=lambda x: -x[0])
        self.asks = sorted(asks, key=lambda x: x[0])
    def imbalance(self):
        bs = self.bids[0][1]; as_ = self.asks[0][1]
        return (bs - as_) / (bs + as_)
```

</details>

### 4. PositionTracker: estado interno

<sub>🔵 si vamos bien · ~6 min</sub>

Define `PositionTracker` con `_cash=0` y `_position=0` (implementación interna) y `apply_fill(fill)` que sume `fill.cash_flow()` a la caja y `fill.size` (con signo) a la posición.

> 💡 El guión bajo dice 'tócalo con métodos, no a mano'.

<sub>practicas: encapsulación + apply_fill</sub>

In [ ]:
class Fill:
    def __init__(self, side, price, size):
        self.side=side; self.price=price; self.size=size
    def cash_flow(self):
        return (-1 if self.side=='buy' else 1)*self.price*self.size
class PositionTracker:
    def __init__(self):
        pass
    def apply_fill(self, fill):
        pass

In [ ]:
# ✅ Comprobación — ejecútala (Shift+Enter). Está plegada a propósito.
assert PositionTracker.__init__.__code__.co_consts != (None,), '⏸ implementa PositionTracker.__init__: su cuerpo sigue siendo pass'
assert PositionTracker.apply_fill.__code__.co_consts != (None,), '⏸ implementa PositionTracker.apply_fill: su cuerpo sigue siendo pass'
t = PositionTracker()
t.apply_fill(Fill('buy',100,0.5))
assert abs(t._cash + 50) < 1e-9 and abs(t._position - 0.5) < 1e-9
print('ok')

<details>
<summary>💭 Pista (antes de mirar la solución)</summary>

En `__init__`, `self._cash = 0.0` y `self._position = 0.0`. En `apply_fill`, `self._cash += fill.cash_flow()`; y la posición suma `fill.size` si es compra, lo resta si es venta.

</details>

<details>
<summary>💡 Ver solución</summary>

```python
class PositionTracker:
    def __init__(self):
        self._cash = 0.0
        self._position = 0.0
    def apply_fill(self, fill):
        self._cash += fill.cash_flow()
        self._position += fill.size if fill.side=='buy' else -fill.size
```

</details>

### 5. equity a mercado

<sub>🔵 si vamos bien · ~6 min</sub>

Reescribe `PositionTracker` añadiendo `equity(mark_price)` = `_cash + _position * mark_price`.

<sub>practicas: componer el estado</sub>

In [ ]:
class Fill:
    def __init__(self, side, price, size):
        self.side=side; self.price=price; self.size=size
    def cash_flow(self):
        return (-1 if self.side=='buy' else 1)*self.price*self.size
class PositionTracker:
    def __init__(self):
        self._cash = 0.0
        self._position = 0.0
    def apply_fill(self, fill):
        self._cash += fill.cash_flow()
        self._position += fill.size if fill.side=='buy' else -fill.size
    def equity(self, mark_price):
        pass

In [ ]:
# ✅ Comprobación — ejecútala (Shift+Enter). Está plegada a propósito.
assert PositionTracker.equity.__code__.co_consts != (None,), '⏸ implementa PositionTracker.equity: su cuerpo sigue siendo pass'
t = PositionTracker()
t.apply_fill(Fill('buy',100,1))
assert abs(t.equity(110) - 10) < 1e-9, 'compra a 100, marca a 110 -> equity 10'
print('ok')

<details>
<summary>💭 Pista (antes de mirar la solución)</summary>

Copia la clase del ejercicio anterior tal cual y añade un método más: `def equity(self, mark_price): return self._cash + self._position * mark_price`.

</details>

<details>
<summary>💡 Ver solución</summary>

```python
class PositionTracker:
    def __init__(self):
        self._cash = 0.0
        self._position = 0.0
    def apply_fill(self, fill):
        self._cash += fill.cash_flow()
        self._position += fill.size if fill.side=='buy' else -fill.size
    def equity(self, mark_price):
        return self._cash + self._position * mark_price
```

</details>

### 6. Los dos objetos, juntos

<sub>🔵 si vamos bien · ~6 min</sub>

Junta las piezas: monta un `OrderBook`, lee su `mid`; crea un `PositionTracker`, aplícale una compra (0.5 @ 100000) y una venta (0.2 @ 100050), y guarda `eq = equity` marcado al `mid` del libro.

> 💡 El equity se marca al `book.mid()`.

<sub>practicas: composición end-to-end</sub>

In [ ]:
class OrderBook:
    def __init__(self, bids, asks):
        self.bids = sorted(bids, key=lambda x: -x[0]); self.asks = sorted(asks, key=lambda x: x[0])
    def best_bid(self): return self.bids[0][0]
    def best_ask(self): return self.asks[0][0]
    def mid(self): return (self.best_bid()+self.best_ask())/2
class Fill:
    def __init__(self, side, price, size):
        self.side=side; self.price=price; self.size=size
    def cash_flow(self):
        return (-1 if self.side=='buy' else 1)*self.price*self.size
class PositionTracker:
    def __init__(self):
        self._cash=0.0; self._position=0.0
    def apply_fill(self, fill):
        self._cash += fill.cash_flow(); self._position += fill.size if fill.side=='buy' else -fill.size
    def equity(self, mark):
        return self._cash + self._position*mark
book = OrderBook([(99990,2.0),(99980,1.0)], [(100010,1.5)])
tracker = PositionTracker()
# aplica los dos fills y guarda eq = equity al mid del libro
eq = None

In [ ]:
# ✅ Comprobación — ejecútala (Shift+Enter). Está plegada a propósito.
assert eq is not None, '⏸ eq sigue en None: completa el ejercicio antes de validar'
assert abs(book.mid() - 100000) < 1e-9
assert abs(eq - 10.0) < 1e-9, 'equity al mid debe ser 10'
print('ok  eq=%.1f' % eq)

<details>
<summary>💭 Pista (antes de mirar la solución)</summary>

Tres pasos: `tracker.apply_fill(Fill('buy', 100000, 0.5))`, lo mismo con la venta, y `eq = tracker.equity(book.mid())`. Fíjate en la firma de `Fill` que te da el given.

</details>

<details>
<summary>💡 Ver solución</summary>

```python
book = OrderBook([(99990,2.0),(99980,1.0)], [(100010,1.5)])
tracker = PositionTracker()
tracker.apply_fill(Fill('buy', 100000, 0.5))
tracker.apply_fill(Fill('sell', 100050, 0.2))
eq = tracker.equity(book.mid())
```

</details>

## Cierre

Composición: un OrderBook contiene niveles; un PositionTracker consume Fills. Los objetos se hablan entre sí.

Si llegas al ejercicio 3 ya tienes el núcleo. Los siguientes y los auxiliares consolidan.

**Siguiente clase:** seguimos construyendo el motor sobre esta pieza.

## 🚀 Llévatelo a un `.py`

Un notebook va genial para explorar, pero el código de verdad vive en archivos `.py` que se ejecutan enteros de una vez. Abre **`book_demo.py`**: es lo que acabas de construir, ordenado y de una pieza.

Ejecútalo desde una terminal:

```bash
python book_demo.py
```

…o aquí mismo, en la siguiente celda:

In [ ]:
!python book_demo.py

> Es la misma pieza que vive en el paquete `exchange/` — aquí, condensada en un archivo que puedes leer de una sentada.